# Setup

## STEP 1:Setup RAG System

- RAG aparameters : k , embedding type, hybrid rag, vector store, document type and quality

In [ ]:
# !pip install langchain==0.1.12
# !pip install sentence-transformers

In [1]:
from langchain.docstore.document import Document
import os
import re
import streamlit as st
from itertools import permutations

db_id_dict = {}
def init_document():
    db_id_dict = {}
    documents = []
    ##############################################################################
    ######### CHANGE INPUT RAG FILE FORLDER HERE #################################
    ##############################################################################
    # directory_path = 'C:\Research-Paper\PAPER-WORK-2024\RAG-FILES'
    directory_path = 'C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7'
    # The directory contains documents in format <TABLE_NAME>__<DB_ID>.txt
    files = os.listdir(directory_path)

    for file in files:
        file_path = os.path.join(directory_path, file)
        if os.path.isfile(file_path):
            with open(file_path, 'r') as f:
                print("Reading file => ",file_path)
                match = re.search(r"(.*?)__(.*?)\.txt", file)
                table_name = match.group(1)
                db_id = match.group(2)
                if db_id not in db_id_dict:
                    db_id_dict[db_id] = [table_name]
                else:
                    db_id_dict[db_id].append(table_name)
                documents.append(Document(page_content=f.read(), metadata={"source": "local", "context": db_id,"table name":table_name}))
    print("Number of documents currently used: ", len(documents))
    return documents

### Embeddings from:

https://huggingface.co/spaces/mteb/leaderboard

TAG PAPER: https://arxiv.org/abs/2212.03533

In [2]:
from langchain.embeddings import HuggingFaceEmbeddings # Source: https://medium.com/international-school-of-ai-data-science/implementing-rag-with-langchain-and-hugging-face-28e3ea66c5f7

def init_embedding():
    print("Init embeddings...")
    # Define the path to the pre-trained model you want to use
    # This is a lightweight and fast model with good performance on semantic textual similarity tasks
    model_path = "sentence-transformers/all-MiniLM-L12-v2"

    """
        Currently the above embedding works fine, but in future we can use more embeddings from https://huggingface.co/spaces/mteb/leaderboard
        As the data grows, some of the embeddings we have tested already are below we can test again these on new data and try among these:
            1. sentence-transformers/all-MiniLM-L12-v2 (RANK: 123)
            2. Alibaba-NLP/gte-large-en-v1.5 (RANK: 19)
            3. dunzhang/stella_en_1.5B_v5 (RANK: 3)
            4. Alibaba-NLP/gte-Qwen2-1.5B-instruct (RANK: 13)
            5. intfloat/e5-base-v2 (USED IN TAG PAPER: https://arxiv.org/pdf/2408.14717)
    """

    # Create a dictionary with model configuration options, specifying to use the CPU for computations
    model_kwargs = {'device': 'cpu'}
    #model_kwargs = {'device': 'cpu', 'trust_remote_code': True} # Sometime this config works for other embeddings in list above

    # Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
    encode_kwargs = {'normalize_embeddings': False}

    # Initialize an instance of HuggingFaceEmbeddings with the specified parameters
    embeddings = HuggingFaceEmbeddings(
        model_name=model_path,  # Provide the pre-trained model's path
        model_kwargs=model_kwargs,  # Pass the model configuration options
        encode_kwargs=encode_kwargs  # Pass the encoding options
    )
    print("Loaded embedding : ",model_path)
    return embeddings

In [3]:
from langchain.vectorstores import FAISS
# Global variable to store the cached result
cached_db = None
cached_embed = None

def init_database():
    # print("Init database...")
    global cached_db
    # print("CACHED_DB = ",cached_db)
    global cached_embed
    # print("EMBED = ",cached_embed)

    if cached_embed is None:
        print("Loading Embedding...")
        cached_embed = init_embedding()

    if True:
        # If the result is already cached, return it
        if cached_db is not None:
            print("Using cached database...")
            return cached_db
        documents = init_document()
        db = FAISS.from_documents(documents, cached_embed)
        # Cache the result
        cached_db = db
        return db

In [4]:
def similarity_k_search(question,k=3):
    db = init_database()
    searchDocs = db.similarity_search_with_score(question,k=k)
    # get a list with page content and score
    for i in range(k):
        print(searchDocs[i][0].metadata["table name"]," from db_id : ",searchDocs[i][0].metadata["context"]," scored ==>",searchDocs[i][1])
    return [(searchDocs[i][0].page_content,searchDocs[i][1],searchDocs[i][0].metadata["table name"],searchDocs[i][0].metadata["context"]) for i in range(k)]

In [5]:
question = "How many farms are there?"
answer = similarity_k_search(question,k=3)

Loading Embedding...
Init embeddings...


C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\t

Loaded embedding :  sentence-transformers/all-MiniLM-L12-v2
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Attribute_Definitions__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\bank__loan_1.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalogs__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalog_Contents_Additional_Attributes__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalog_Contents__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalog_Structure__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\church__wedding.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\city__farm.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Claims__insurance_policies.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data

In [6]:
answer

[('/*\nThe farm table contains detailed data about the livestock population and farm resources across multiple years. The table tracks the total number of different types of animals on the farm and distinguishes between various categories such as horses, cattle, oxen, bulls, cows, pigs, and sheep/goats.\n\nFarm_ID\tA unique identifier for each farm entry.\nYear\tThe year for which the data is recorded.\nTotal_Horses\tThe total number of horses on the farm.\nWorking_Horses\tThe number of horses used for working purposes on the farm.\nTotal_Cattle\tThe total number of cattle (oxen, bulls, and cows) on the farm.\nOxen\tThe number of oxen specifically on the farm.\nBulls\tThe number of bulls specifically on the farm.\nCows\tThe number of cows specifically on the farm.\nPigs\tThe total number of pigs on the farm.\nSheep_and_Goats\tThe total number of sheep and goats on the farm.\n*/\n\nCREATE TABLE "farm" (\n"Farm_ID" int,\n"Year" int,\n"Total_Horses" real,\n"Working_Horses" real,\n"Total_C

## Step 2: Read relevant queries from dataset

In [7]:
import json
import pandas as pd

##############################################################################################
######### CHANGE SPIDER DATA TRAIN SPIDER.JSON LOCATION HERE #################################
##############################################################################################
# Path to the JSON file
file_path = r"C:\Research-Paper\spider_data\spider_data\train_spider.json"

# Read the JSON file
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# Convert to a Pandas DataFrame
df = pd.json_normalize(data)

# Display the DataFrame
df.head()

,db_id,query,query_toks,query_toks_no_value,question,question_toks,sql.from.table_units,sql.from.conds,sql.select,sql.where,...,sql.except.from.conds,sql.except.select,sql.except.where,sql.except.groupBy,sql.except.having,sql.except.orderBy,sql.except.limit,sql.except.intersect,sql.except.union,sql.except.except
0,department_management,SELECT count(*) FROM head WHERE age > 56,"[SELECT, count, (, *, ), FROM, head, WHERE, ag...","[select, count, (, *, ), from, head, where, ag...",How many heads of the departments are older th...,"[How, many, heads, of, the, departments, are, ...","[[table_unit, 1]]",[],"[False, [[3, [0, [0, 0, False], None]]]]","[[False, 3, [0, [0, 10, False], None], 56.0, N...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,department_management,"SELECT name , born_state , age FROM head ORD...","[SELECT, name, ,, born_state, ,, age, FROM, he...","[select, name, ,, born_state, ,, age, from, he...","List the name, born state and age of the heads...","[List, the, name, ,, born, state, and, age, of...","[[table_unit, 1]]",[],"[False, [[0, [0, [0, 8, False], None]], [0, [0...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,department_management,"SELECT creation , name , budget_in_billions ...","[SELECT, creation, ,, name, ,, budget_in_billi...","[select, creation, ,, name, ,, budget_in_billi...","List the creation year, name and budget of eac...","[List, the, creation, year, ,, name, and, budg...","[[table_unit, 0]]",[],"[False, [[0, [0, [0, 3, False], None]], [0, [0...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,department_management,"SELECT max(budget_in_billions) , min(budget_i...","[SELECT, max, (, budget_in_billions, ), ,, min...","[select, max, (, budget_in_billions, ), ,, min...",What are the maximum and minimum budget of the...,"[What, are, the, maximum, and, minimum, budget...","[[table_unit, 0]]",[],"[False, [[1, [0, [0, 5, False], None]], [2, [0...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,department_management,SELECT avg(num_employees) FROM department WHER...,"[SELECT, avg, (, num_employees, ), FROM, depar...","[select, avg, (, num_employees, ), from, depar...",What is the average number of employees of the...,"[What, is, the, average, number, of, employees...","[[table_unit, 0]]",[],"[False, [[5, [0, [0, 6, False], None]]]]","[[False, 1, [0, [0, 4, False], None], 10.0, 15...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(df.shape)

(7000, 50)


In [9]:
spider_df = df[["db_id","query","question"]]

In [10]:
#################################################################################
######### CHANGE DB_ID LIST HERE (CURRENT : 15) #################################
#################################################################################

# List of db_id values to match
l = ["farm","film_rank","election","wrestler","wedding","swimming","climbing","device","loan_1","movie_1",
    "railway","coffee_shop","game_1","insurance_policies","product_catalog"]

# Filtering rows where db_id is in the list l
filtered_spider_df = spider_df[spider_df['db_id'].isin(l)]

# Counting occurrences of each element in l
count_per_element = filtered_spider_df['db_id'].value_counts()

# Ensuring all elements from l are in the count
count_per_element = count_per_element.reindex(l, fill_value=0)

print("Filtered DataFrame:")
print(filtered_spider_df.shape)
print("\n---------------\nCount of each element:")
print(count_per_element)

Filtered DataFrame:
(719, 3)

---------------
Count of each element:
db_id
farm                  40
film_rank             48
election              68
wrestler              40
wedding               20
swimming              30
climbing              40
device                40
loan_1                80
movie_1               98
railway               21
coffee_shop           18
game_1                86
insurance_policies    48
product_catalog       42
Name: count, dtype: int64


In [11]:
filtered_spider_df.columns = ['db_id', 'spider_query', 'question']
filtered_spider_df.head()

,db_id,spider_query,question
16,farm,SELECT count(*) FROM farm,How many farms are there?
17,farm,SELECT count(*) FROM farm,Count the number of farms.
18,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...
19,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,..."
20,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...


In [13]:
filtered_spider_df.shape

(719, 3)

## Step 3:  Setup SQLCoder (Text2SQL)

In [14]:
import os

def configure_aws(access_key_id, secret_access_key, region, profile):
    # Set environment variables for AWS configuration
    os.environ['AWS_ACCESS_KEY_ID'] = access_key_id
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_access_key
    os.environ['AWS_DEFAULT_REGION'] = region

    print("AWS configuration has been set successfully for : ",profile)


id_endpoint_name = "<ENDPOINT>" 
instance = "llama3 SqlCoder"

id_config_data = {'profile':'<PROFILE>', 
                     'access_key_id' :'<ACCESS_KEY_ID>',
                     'secret_access_key' :'<SECRET_ACCESS_KEY>',
                     'region' :'<REGION>',
                     'eprid_endpoint_name':id_endpoint_name}


# SETUP AWS
configure_aws(access_key_id = id_config_data['access_key_id'], 
              secret_access_key = id_config_data['secret_access_key'], 
              region = id_config_data['region'], 
              profile = id_config_data['profile'])

AWS configuration has been set successfully for :  EPRID


In [15]:
import json
from langchain.llms.sagemaker_endpoint import LLMContentHandler
import streamlit as st
from langchain import SagemakerEndpoint
from langchain import LLMChain
from langchain.prompts import PromptTemplate

class Llama3ContentHandler(LLMContentHandler):
    content_type = "application/json"
    accepts = "application/json"

    def transform_input(self, prompt: str, model_kwargs: dict) -> bytes:
        self.len_prompt = len(prompt)
        input_dict = {
            "inputs": prompt,
            "parameters": model_kwargs
        }
        input_str = json.dumps(input_dict)
        return input_str.encode('utf-8')

    def transform_output(self, output: bytes) -> str:
        response_json = output.read()
        res = json.loads(response_json)
        if type(res) is list:
            return res[0]['generated_text']
        else:
            return res['generated_text']
 
   
def change_llm(llm_chain, instance, endpoint_name, prompt_template, input_variables=None,
               content_handler=Llama3ContentHandler(), max_new_tokens=1024, repetition_penalty=1.1,
               return_full_text=True, stop=None, temperature=None, top_p=None,region=None):
    if input_variables is None:
        input_variables = ["question"]
    llm_chain['instance'] = instance
    llm_chain['endpoint_name'] = endpoint_name
    prompt = PromptTemplate(
        input_variables=input_variables, template=prompt_template
    )

    llm_chain['prompt_template'] = prompt_template
    llm_chain['prompt'] = prompt
    llm_chain['content_handler'] = content_handler

    if stop:
        model_kwargs = {
            "max_new_tokens": max_new_tokens,
            "top_p": top_p,
            "temperature": temperature,
            "repetition_penalty": repetition_penalty,
            "return_full_text": return_full_text,
            "stop": stop
        }
    else:
        model_kwargs = {
            "max_new_tokens": max_new_tokens,
            "top_p": top_p,
            "temperature": temperature,
            "repetition_penalty": repetition_penalty,
            "return_full_text": return_full_text
        }

    llm = SagemakerEndpoint(
        endpoint_name=llm_chain['endpoint_name'],
        region_name=region,
        model_kwargs=model_kwargs,
        endpoint_kwargs={"CustomAttributes": 'accept_eula=true'},
        content_handler=content_handler
    )
    chain = LLMChain(llm=llm, prompt=prompt, verbose=True)  # Verbose turned on

    # Push all common references to the dict
    llm_chain['llm'] = llm
    llm_chain['chain'] = chain

    return llm_chain

In [29]:
PREFIX_PROMPT = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `{question}`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
- If you are fetching data from a table only then use its columns to filter out the data.
- You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDDL statements:\n\n"""


PREFIX_PROMPT = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `{question}`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column unless not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- Avoid using SQL Aliases if not required in query.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
- If you are fetching data from a table only then use its columns to filter out the data.
- You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDDL statements:\n\n"""


SUFFIX_PROMPT = """<|eot_id|><|start_header_id|>assistant<|end_header_id|>
The following SQL query best answers the question `{question}`:
```sql
"""

In [17]:
question = "How many farms are there?"
answer = similarity_k_search(question,k=3)
answer

Using cached database...
farm  from db_id :  farm  scored ==> 0.98307145
farm_competition  from db_id :  farm  scored ==> 1.4487716
competition_record  from db_id :  farm  scored ==> 1.4860706


[('/*\nThe farm table contains detailed data about the livestock population and farm resources across multiple years. The table tracks the total number of different types of animals on the farm and distinguishes between various categories such as horses, cattle, oxen, bulls, cows, pigs, and sheep/goats.\n\nFarm_ID\tA unique identifier for each farm entry.\nYear\tThe year for which the data is recorded.\nTotal_Horses\tThe total number of horses on the farm.\nWorking_Horses\tThe number of horses used for working purposes on the farm.\nTotal_Cattle\tThe total number of cattle (oxen, bulls, and cows) on the farm.\nOxen\tThe number of oxen specifically on the farm.\nBulls\tThe number of bulls specifically on the farm.\nCows\tThe number of cows specifically on the farm.\nPigs\tThe total number of pigs on the farm.\nSheep_and_Goats\tThe total number of sheep and goats on the farm.\n*/\n\nCREATE TABLE "farm" (\n"Farm_ID" int,\n"Year" int,\n"Total_Horses" real,\n"Working_Horses" real,\n"Total_C

In [18]:
TABLE_INFO = ""

for table in answer:
    TABLE_INFO = TABLE_INFO + table[0]+"\n\n"
    
TABLE_INFO = TABLE_INFO.rstrip()
prompt = (PREFIX_PROMPT + TABLE_INFO + SUFFIX_PROMPT)
print(prompt)

<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `{question}`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column unless not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- Avoid using SQL Aliases if not required in query.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
- If you are fet

In [31]:
llm_chain = {}

content_handler = Llama3ContentHandler()
return_full_text = False
stop = ""
temperature=0.01
top_p=0.7
max_new_tokens = 1024
return_full_text = False
prompt_template=prompt

llm_chain = change_llm(llm_chain, instance, id_endpoint_name, prompt_template,
                           input_variables=["question"],
                           content_handler=content_handler, max_new_tokens=max_new_tokens, repetition_penalty=0.95,
                           return_full_text=return_full_text, stop=stop,temperature=temperature,top_p=top_p,region=id_config_data['region'])

In [20]:
question = "How many farms are there?"
sqlcoder_generated_query = llm_chain['chain'].run({"question": question})
print(sqlcoder_generated_query)

C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(




> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `How many farms are there?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column unless not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- Avoid using SQL Aliases if not required in query.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML state

In [21]:
print(sqlcoder_generated_query)

SELECT COUNT(DISTINCT f.Farm_ID) FROM "farm" f;


## Step 4:  Experiment

k value is 3, as any sql query wont have or use more than 3 tables

In [22]:
sqlcoder_generated_queries = []
llm_chain = {}
content_handler = Llama3ContentHandler()
return_full_text = False
stop = ""
temperature=0.01
top_p=0.7
max_new_tokens = 1024
return_full_text = False
k_value = 3

i = 0
for index, row in filtered_spider_df.iterrows():
    i = i +1
    #if i > 5:
    #    break
    print(f"################################# Row {index} #################################")
    print("Number : ",i)
    print("DB_ID : ",row["db_id"],"\nQUESTION : ",row["question"])
    print("CORRECT SQL SPIDER QUERY : ",row["spider_query"])
    question = row["question"]
    print("*"*50)
    answer = similarity_k_search(question,k=k_value)
    print("*"*50)
    TABLE_INFO = ""

    for table in answer:
        TABLE_INFO = TABLE_INFO + table[0]+"\n\n"
    
    TABLE_INFO = TABLE_INFO.rstrip()
    prompt = (PREFIX_PROMPT + TABLE_INFO + SUFFIX_PROMPT)
    prompt_template=prompt
    
    llm_chain = {}
    llm_chain = change_llm(llm_chain, instance, id_endpoint_name, prompt_template,
                           input_variables=["question"],
                           content_handler=content_handler, max_new_tokens=max_new_tokens, repetition_penalty=0.95,
                           return_full_text=return_full_text, stop=stop,temperature=temperature,top_p=top_p,region=id_config_data['region'])
    sqlcoder_generated_query = llm_chain['chain'].run({"question": question})
    print("TEXT2SQL GENERATED QUERY : ",sqlcoder_generated_query)
    sqlcoder_generated_queries.append(sqlcoder_generated_query)
    

################################# Row 16 #################################
Number :  1
DB_ID :  farm 
QUESTION :  How many farms are there?
CORRECT SQL SPIDER QUERY :  SELECT count(*) FROM farm
**************************************************
Using cached database...
farm  from db_id :  farm  scored ==> 0.98307145
farm_competition  from db_id :  farm  scored ==> 1.4487716
competition_record  from db_id :  farm  scored ==> 1.4860706
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `How many farms are there?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any 


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT COUNT(DISTINCT f."Farm_ID") FROM "farm" f;
################################# Row 18 #################################
Number :  3
DB_ID :  farm 
QUESTION :  List the total number of horses on farms in ascending order.
CORRECT SQL SPIDER QUERY :  SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC
**************************************************
Using cached database...
farm  from db_id :  farm  scored ==> 0.66637516
competition_record  from db_id :  farm  scored ==> 1.2602258
farm_competition  from db_id :  farm  scored ==> 1.4009414
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `List the total number of horses on farms in ascending order.`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the 


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT f.Farm_ID, f.Total_Horses FROM farm f ORDER BY f.Total_Horses ASC;
################################# Row 20 #################################
Number :  5
DB_ID :  farm 
QUESTION :  What are the hosts of competitions whose theme is not "Aliens"?
CORRECT SQL SPIDER QUERY :  SELECT Hosts FROM farm_competition WHERE Theme !=  'Aliens'
**************************************************
Using cached database...
farm_competition  from db_id :  farm  scored ==> 1.0501661
stadium  from db_id :  swimming  scored ==> 1.3983935
competition_record  from db_id :  farm  scored ==> 1.4186804
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `What are the hosts of competitions whose theme is not "Aliens"?`

### Instructions
- Given an input question, create a syntactically correct query 


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT fc.Hosts FROM farm_competition fc WHERE fc.Theme!= 'Aliens';
################################# Row 22 #################################
Number :  7
DB_ID :  farm 
QUESTION :  What are the themes of farm competitions sorted by year in ascending order?
CORRECT SQL SPIDER QUERY :  SELECT Theme FROM farm_competition ORDER BY YEAR ASC
**************************************************
Using cached database...
farm_competition  from db_id :  farm  scored ==> 0.93417525
competition_record  from db_id :  farm  scored ==> 1.0060741
farm  from db_id :  farm  scored ==> 1.2381489
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `What are the themes of farm competitions sorted by year in ascending order?`

### Instructions
- Given an input question, create a syntactically correct q


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT fc."Year", fc."Theme" FROM "farm_competition" fc ORDER BY fc."Year" ASC;
################################# Row 24 #################################
Number :  9
DB_ID :  farm 
QUESTION :  What is the average number of working horses of farms with more than 5000 total number of horses?
CORRECT SQL SPIDER QUERY :  SELECT avg(Working_Horses) FROM farm WHERE Total_Horses  >  5000
**************************************************
Using cached database...
farm  from db_id :  farm  scored ==> 0.74032784
competition_record  from db_id :  farm  scored ==> 1.3772876
farm_competition  from db_id :  farm  scored ==> 1.4861748
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `What is the average number of working horses of farms with more than 5000 total number of horses?`

### Inst

...


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT v.GName FROM Video_Games v JOIN Plays_Games pg ON v.GameID = pg.GameID WHERE pg.Hours_Played >= 1000;
################################# Row 6047 #################################
Number :  712
DB_ID :  game_1 
QUESTION :  Show all game names played by Linda Smith
CORRECT SQL SPIDER QUERY :  SELECT Gname FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.gameid  =  T2.gameid JOIN Student AS T3 ON T3.Stuid  =  T1.Stuid WHERE T3.Lname  =  "Smith" AND T3.Fname  =  "Linda"
**************************************************
Using cached database...
Plays_Games  from db_id :  game_1  scored ==> 1.2520833
Video_Games  from db_id :  game_1  scored ==> 1.2629893
SportsInfo  from db_id :  game_1  scored ==> 1.3905296
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `Show all game


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT v.GName FROM Plays_Games pg JOIN Student s ON pg.StuID = s.StuID JOIN Video_Games v ON pg.GameID = v.GameID WHERE s.Name = 'Linda Smith';
################################# Row 6049 #################################
Number :  714
DB_ID :  game_1 
QUESTION :  Find the last and first name of students who are playing Football or Lacrosse.
CORRECT SQL SPIDER QUERY :  SELECT T2.lname ,  T2.fname FROM SportsInfo AS T1 JOIN Student AS T2 ON T1.StuID  =  T2.StuID WHERE T1.SportName  =  "Football" OR T1.SportName  =  "Lacrosse"
**************************************************
Using cached database...
SportsInfo  from db_id :  game_1  scored ==> 0.89947695
Student  from db_id :  game_1  scored ==> 1.0149732
stadium  from db_id :  swimming  scored ==> 1.2391326
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL 


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT s.Fname, s.LName FROM Student s JOIN SportsInfo si ON s.StuID = si.StuID WHERE si.SportName = 'Football' OR si.SportName = 'Lacrosse' ORDER BY s.Fname, s.LName;
################################# Row 6051 #################################
Number :  716
DB_ID :  game_1 
QUESTION :  Find the first name and age of the students who are playing both Football and Lacrosse.
CORRECT SQL SPIDER QUERY :  SELECT fname ,  age FROM Student WHERE StuID IN (SELECT StuID FROM Sportsinfo WHERE SportName  =  "Football" INTERSECT SELECT StuID FROM Sportsinfo WHERE SportName  =  "Lacrosse")
**************************************************
Using cached database...
SportsInfo  from db_id :  game_1  scored ==> 1.0123577
Student  from db_id :  game_1  scored ==> 1.1304641
stadium  from db_id :  swimming  scored ==> 1.2896695
**************************************************


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|st


> Finished chain.
TEXT2SQL GENERATED QUERY :  SELECT s.Fname, s.Age FROM Student s JOIN SportsInfo si ON s.StuID = si.StuID WHERE si.SportName IN ('Football', 'Lacrosse') GROUP BY s.Fname, s.Age HAVING COUNT(DISTINCT si.SportName) = 2;
################################# Row 6053 #################################
Number :  718
DB_ID :  game_1 
QUESTION :  Find the last name and gender of the students who are playing both Call of Destiny and Works of Widenius games.
CORRECT SQL SPIDER QUERY :  SELECT lname ,  sex FROM Student WHERE StuID IN (SELECT T1.StuID FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.GameID  =  T2.GameID WHERE T2.Gname  =  "Call of Destiny" INTERSECT SELECT T1.StuID FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.GameID  =  T2.GameID WHERE T2.Gname  =  "Works of Widenius")
**************************************************
Using cached database...
Plays_Games  from db_id :  game_1  scored ==> 1.0796599
Student  from db_id :  game_1  scored ==> 1.1669419
Sport



> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `what is the last name and gender of all students who played both Call of Destiny and Works of Widenius?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column unless not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- Avoid using SQL Aliases if not required in query.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same

In [23]:
len(sqlcoder_generated_queries)

719

In [24]:
sqlcoder_generated_queries

['SELECT COUNT(DISTINCT f.Farm_ID) FROM "farm" f;',
 'SELECT COUNT(DISTINCT f."Farm_ID") FROM "farm" f;',
 'SELECT f."Farm_ID", f."Year", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;',
 'SELECT f.Farm_ID, f.Total_Horses FROM farm f ORDER BY f.Total_Horses ASC;',
 "SELECT DISTINCT fc.Hosts FROM farm_competition fc WHERE fc.Theme!= 'Aliens';",
 "SELECT fc.Hosts FROM farm_competition fc WHERE fc.Theme!= 'Aliens';",
 'SELECT fc.Year, fc.Theme FROM farm_competition fc ORDER BY fc.Year ASC;',
 'SELECT fc."Year", fc."Theme" FROM "farm_competition" fc ORDER BY fc."Year" ASC;',
 'SELECT AVG(f.Working_Horses) AS average_working_horses FROM farm f WHERE f.Total_Horses > 5000;',
 'SELECT AVG(f.Working_Horses) AS average_working_horses FROM farm f WHERE f.Total_Horses > 5000;',
 'SELECT MAX(f.Cows) AS Max_Cows, MIN(f.Cows) AS Min_Cows FROM farm f;',
 'SELECT MAX(f.Cows) AS Max_Cows, MIN(f.Cows) AS Min_Cows FROM farm f;',
 'SELECT COUNT(DISTINCT c.Status) FROM "city" c;',
 'SELECT C

In [68]:
#filtered_spider_df_2 = filtered_spider_df.head()

In [25]:
filtered_spider_df.shape

(719, 3)

In [26]:
filtered_spider_df['text2sql_query'] = sqlcoder_generated_queries

C:\Users\GuPr564\AppData\Local\Temp\ipykernel_34004\1698195345.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_spider_df['text2sql_query'] = sqlcoder_generated_queries


In [27]:
filtered_spider_df

,db_id,spider_query,question,text2sql_query
16,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;"
17,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;"
18,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""..."
19,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O..."
20,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...
...,...,...,...,...
6050,game_1,"SELECT T2.lname , T2.fname FROM SportsInfo AS...",What is the first and last name of all student...,"SELECT s.Fname, s.LName FROM Student s JOIN Sp..."
6051,game_1,"SELECT fname , age FROM Student WHERE StuID I...",Find the first name and age of the students wh...,"SELECT s.Fname, s.Age FROM Student s JOIN Spor..."
6052,game_1,"SELECT fname , age FROM Student WHERE StuID I...",What are the first names and ages of all stude...,"SELECT s.Fname, s.Age FROM Student s JOIN Spor..."
6053,game_1,"SELECT lname , sex FROM Student WHERE StuID I...",Find the last name and gender of the students ...,"SELECT s.LName, s.Sex FROM Student s JOIN Play..."


In [28]:
import pandas as pd

# Save the DataFrame to an Excel file
########################################################################
######### CHANGE OUTPUT FOLDER HERE ####################################
########################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7.xlsx"
filtered_spider_df.to_excel(file_path, index=False)

print(f"DataFrame saved successfully to {file_path}")

DataFrame saved successfully to C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7.xlsx
